# Mesh 9 · From a scan to a mesh, and back to the design

**Application, advanced.** Companion to [Week 05 §12](../README.md#12-mesh-in-architecture-four-applications)
and `mesh_applications/scan_to_mesh.py`. Uses notebooks 01–04.

![A point cloud with outliers, its trimmed Delaunay mesh with one hole, and the deviation from the design surface as a heat map](../figures/app-scan.svg)

A roof has been built and laser-scanned. The scan is 925 points: no faces,
no neighbours, some of them wrong. The question from site is simple:
**where is the built roof more than 20 mm from the drawing?** To answer it
the points have to become a surface, and every step on the way uses
something from the fundamentals.

| Step | Idea it leans on |
| --- | --- |
| 1 · look at the raw points | a point cloud is not a mesh (01 §1) |
| 2 · drop the outliers | neighbours, without a mesh yet |
| 3 · triangulate | `from_vertices_and_faces` (01 §6) |
| 4 · trim what is not surface | `delete_face`, and Euler as the judge (04 §3) |
| 5 · smooth the noise | move vertices, topology stays (03 §2) |
| 6 · compare with the design | attributes, queries, colour (03 §3–5) |
| 7 · report | areas per vertex (03 §1) |

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in its lecture folder or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

import math

from compas.datastructures import Mesh
from compas.geometry import KDTree, delaunay_triangulation

# the scan itself comes from the lecture script: it is the "given" here
sys.path.insert(0, str(NB.parent / "Lecture_05" / "mesh_applications"))
from scan_to_mesh import design_surface, simulate_scan

## 1 · What the scanner gives you

`simulate_scan()` stands in for the instrument: points over a 10 m × 8 m
roof, each a few centimetres off (noise), a round patch with no points at
all (a rooflight the laser went through), and 25 points that are simply
wrong (a bird, a reflection). `design_surface(x, y)` is the roof as drawn.

In [ ]:
raw = simulate_scan()
print(len(raw), "points; the first three:")
for point in raw[:3]:
    print("  ", [round(c, 3) for c in point])
zs = [p[2] for p in raw]
print(f"heights from {min(zs):.2f} to {max(zs):.2f} m")
draw_points(raw, "the scan, from above", size=(560, 440))

From above it looks like a roof with a hole. But it is a **list**: point
17 has no idea which points are near it. Before meshing, the wrong points
have to go, or they become spikes.

## 2 · Outliers: a point should agree with its neighbours

No mesh yet, so "neighbours" means *nearest in space*. A `KDTree` answers
that quickly. For one point: find its 8 nearest, take the **median** of
their heights, and see how far the point is from it.

In [ ]:
tree = KDTree(raw)


def height_gap(point, k=8):
    """How far a point's height is from the median height of its k nearest neighbours."""
    nearest = tree.nearest_neighbors(point, k + 1, distance_sort=True)[1:]      # [0] is the point itself
    heights = sorted(raw[index][2] for _, index, _ in nearest)
    return abs(point[2] - heights[len(heights) // 2])


for i in (0, 1, len(raw) - 1):
    print(f"point {i:3d}: {height_gap(raw[i]) * 1000:6.0f} mm from its neighbours' median")

Two points within the scanner's noise, one more than half a metre out.
The median matters: one wild neighbour cannot drag it, where it would drag
an average.

In [ ]:
gaps = [height_gap(p) for p in raw]
clean = [p for p, gap in zip(raw, gaps) if gap < 0.2]
dropped = [p for p, gap in zip(raw, gaps) if gap >= 0.2]
print(f"kept {len(clean)}, dropped {len(dropped)}")
draw_points(raw, "dropped points in red", size=(560, 440),
            colors=["#e03131" if gap >= 0.2 else "#ced4da" for gap in gaps])

## 3 · Triangulate: now it has faces

Delaunay triangulation connects points in plan into well-shaped triangles
(Week 10 builds it from scratch). It returns faces as index triples: the
points list and those triples are exactly what `from_vertices_and_faces`
wants.

In [ ]:
faces = delaunay_triangulation(clean)
everything = Mesh.from_vertices_and_faces(clean, [[int(i) for i in face] for face in faces])
print("V E F =", everything.number_of_vertices(), everything.number_of_edges(), everything.number_of_faces())
print("euler:", everything.euler())
draw_mesh(everything, "Delaunay: every point connected", vertex_labels=False, size=(560, 440))

Euler is **1: a sheet with no hole**. But the roof has a rooflight. Look
at the picture: the triangulation has stretched long thin triangles
straight across it, and along the ragged edges too. Delaunay fills the
convex outline of whatever it is given. Those triangles are not roof.
They are the *absence of data*, drawn as if it were surface.

## 4 · Trim, and let Euler be the judge

Real surface triangles are small, because the scan is dense. Invented ones
are long. So: delete every face with an edge longer than some limit.

**How long is too long?** Guess too high and the rooflight stays covered.
Too low, and you punch false holes wherever the scan was a little sparse.
You know the right answer in advance: one hole, so `V − E + F` must be 0.

In [ ]:
def trimmed(mesh, max_edge):
    result = mesh.copy()
    for face in list(result.faces()):
        corners = result.face_vertices(face)
        if any(result.edge_length((u, v)) > max_edge for u, v in zip(corners, corners[1:] + corners[:1])):
            result.delete_face(face)
    result.remove_unused_vertices()
    return result


print("max edge   faces   euler")
for limit in (0.6, 0.7, 0.8, 0.9, 1.0, 1.4, 2.0, 3.0):
    m = trimmed(everything, limit)
    verdict = "<- one hole: correct" if m.euler() == 0 else ""
    print(f"  {limit:4.1f} m   {m.number_of_faces():5d}   {m.euler():4d}   {verdict}")

One integer per row, and it sorts the thresholds into three groups without
anyone looking at a picture: too strict (negative: many false holes), right
(0), too loose (1: the rooflight is roofed over again).

In [ ]:
roof = trimmed(everything, 1.4)
too_strict = trimmed(everything, 0.7)
row(draw_mesh(too_strict, f"max edge 0.7 m: euler {too_strict.euler()}", vertex_labels=False, size=(450, 360),
              highlight_edges=too_strict.edges_on_boundary()),
    draw_mesh(roof, f"max edge 1.4 m: euler {roof.euler()}", vertex_labels=False, size=(450, 360),
              highlight_edges=roof.edges_on_boundary()))

The boundary (orange) is now the roof's real edge plus one ring round the
rooflight. That ring is a list of vertices you could hand to whoever
details the kerb.

## 5 · Smooth the noise, keep the shape

Each height is a few centimetres off. Average each vertex with its
neighbours and independent errors cancel. Only `z` moves: the plan
position of a scan point is trustworthy, and the mesh's faces are not
touched (notebook 03 §2).

In [ ]:
def deviations(mesh):
    """Scan minus design, at every vertex, in metres."""
    return {k: mesh.vertex_coordinates(k)[2] - design_surface(*mesh.vertex_coordinates(k)[:2]) for k in mesh.vertices()}


def rms(values):
    values = list(values)
    return math.sqrt(sum(v * v for v in values) / len(values))


def smoothed(mesh, weight=0.5):
    result = mesh.copy()
    for k in mesh.vertices():
        around = mesh.vertex_neighbors(k)
        average = sum(mesh.vertex_attribute(n, "z") for n in around) / len(around)
        result.vertex_attribute(k, "z", (1 - weight) * mesh.vertex_attribute(k, "z") + weight * average)
    return result


history = [rms(deviations(roof).values())]
current = roof
for _ in range(8):
    current = smoothed(current)
    history.append(rms(deviations(current).values()))
print("RMS distance from the design, in mm, after 0..8 rounds:")
print("  ", [round(v * 1000, 1) for v in history])
chart({"RMS deviation (mm)": [v * 1000 for v in history]}, "smoothing: better, then worse", xlabel="rounds")

It improves, then turns round. The first rounds remove **noise**. Keep
going and the averaging starts to flatten the roof's real curvature: it
removes **shape**, and the mesh drifts away from the design again. More
smoothing is not more accurate. (In a real job there is no design surface
to tell you where the turn is; two rounds is a cautious choice, and Week
10 has a smoother that does not shrink.)

In [ ]:
best = min(range(len(history)), key=history.__getitem__)
print("lowest after", best, "rounds; we keep 2")
roof = smoothed(smoothed(roof))

## 6 · Back to the design: deviation as an attribute

Store the difference on every vertex. Then the site question is a query.

In [ ]:
for k, d in deviations(roof).items():
    roof.vertex_attribute(k, "deviation", d)

tolerance = 0.020
high = list(roof.vertices_where(deviation=(tolerance, 9)))
low = list(roof.vertices_where(deviation=(-9, -tolerance)))
print(f"{roof.number_of_vertices()} vertices: {len(high)} more than 20 mm high, {len(low)} more than 20 mm low")
worst = max(roof.vertices(), key=lambda k: abs(roof.vertex_attribute(k, "deviation")))
x, y, z = roof.vertex_coordinates(worst)
print(f"worst: {roof.vertex_attribute(worst, 'deviation') * 1000:+.0f} mm at x = {x:.1f}, y = {y:.1f}")

In [ ]:
def face_deviation(mesh, face):
    return sum(mesh.vertex_attribute(k, "deviation") for k in mesh.face_vertices(face)) / 3


limit = max(abs(d) for d in roof.vertices_attribute("deviation"))
colours = {face: ramp(0.5 + 0.5 * face_deviation(roof, face) / limit) for face in roof.faces()}
draw_mesh(roof, "built minus drawn: blue low, orange high", vertex_labels=False, face_colors=colours, size=(620, 480))

In [ ]:
from compas.colors import Color

view3d((roof, {"facecolor": {face: Color.from_hex(c) for face, c in colours.items()}, "show_edges": False}))

## 7 · The report: how much roof is out of tolerance?

A count of vertices is not what a contractor needs. **Square metres** is. Each
vertex stands for its share of the surface (`vertex_area`, notebook 03
§1), so the area out of tolerance is a sum.

In [ ]:
total = roof.area()
out = sum(roof.vertex_area(k) for k in high + low)
print(f"roof scanned:      {total:5.1f} m2")
print(f"out of tolerance:  {out:5.1f} m2  ({out / total:.0%})")
print(f"  too high:        {sum(roof.vertex_area(k) for k in high):5.1f} m2")
print(f"  too low:         {sum(roof.vertex_area(k) for k in low):5.1f} m2")

Be honest about what that number is. The scanner's own noise was 30 mm
per point, and the tolerance is 20 mm. Smoothing brought the noise down,
but part of what is flagged here is still **measurement**, not building.
A report that does not say so will send someone up a ladder to fix a roof
that is fine. The method is sound; the instrument has to be better than
the tolerance it is checking.

## What this used

Outliers by neighbours → faces from points → **Euler to choose a
threshold** → smoothing that moves only `z` → an attribute per vertex → a
query → an area. Seven steps, none longer than ten lines, each one a
fundamental with a job to do.

For scans that are not a height field (a whole room, a statue) see
`compas_cgal.reconstruction`: the idea is the same, the triangulation is
harder.

## Your turn
1. Rerun with `simulate_scan(noise=0.005)`, a better scanner. What happens
   to the area out of tolerance, and to the best number of smoothing
   rounds?
2. Add a second rooflight (`simulate_scan` takes one `hole`; filter the
   raw points yourself). What Euler characteristic is correct now?
3. Replace the median in Step 2 by the mean. How many points are dropped?
   Why?
4. Colour the mesh by **slope** instead of deviation, using
   `face_normal`. Where would water stand on the built roof?